In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import optax
from flax import nnx

from sklearn.datasets import make_regression
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

from deepordinal.jax import OrdinalOutput, ordinal_loss

In [ ]:
x, y = make_regression(n_samples=10000, n_features=8, n_informative=6, random_state=0)
y = QuantileTransformer().fit_transform(y.reshape(-1, 1))[:, 0]
y = np.clip(np.floor(y * 4), 0, 3).astype(np.int32)  # 4 ordinal classes: 0, 1, 2, 3

x_train, x_val, y_train, y_val = train_test_split(
    x.astype(np.float32), y, test_size=0.25, random_state=0
)
print("Class distribution:", np.bincount(y_train))

In [ ]:
class Model(nnx.Module):
    def __init__(self, rngs):
        self.features = nnx.Sequential(nnx.Linear(8, 16, rngs=rngs), jax.nn.relu)
        self.head = OrdinalOutput(input_dim=16, output_dim=4, rngs=rngs)

    def __call__(self, x):  # class probabilities
        return self.head(self.features(x))


model = Model(nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

In [ ]:
def loss_fn(model, xb, yb):
    return ordinal_loss(model.head.logit(model.features(xb)), yb, model.head.interior_thresholds)


@nnx.jit
def train_step(model, optimizer, xb, yb):
    loss, grads = nnx.value_and_grad(loss_fn)(model, xb, yb)
    optimizer.update(model, grads)
    return loss


eval_step = nnx.jit(loss_fn)


def batches(x, y, batch_size=16, rng=None):
    idx = np.arange(len(x)) if rng is None else rng.permutation(len(x))
    for start in range(0, len(x), batch_size):
        yield x[idx[start:start + batch_size]], y[idx[start:start + batch_size]]


rng = np.random.default_rng(0)
epochs = 50
train_losses, val_losses = [], []
for epoch in range(epochs):
    train_losses.append(np.mean([train_step(model, optimizer, xb, yb) for xb, yb in batches(x_train, y_train, rng=rng)]))
    val_losses.append(np.mean([eval_step(model, xb, yb) for xb, yb in batches(x_val, y_val)]))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
preds = np.asarray(model(x_val).argmax(axis=1))
print(f"Validation accuracy: {np.mean(preds == y_val):.4f}")

In [ ]:
plt.plot(train_losses, "--", label="train_loss")
plt.plot(val_losses, label="val_loss")
plt.title("Training and validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()